# Smoke test

One dataset, 200 instances, both tiers, the full routing analysis. Under an
hour on a T4. Run this before committing to the full grid.

It proves five things:

1. label-likelihood scoring returns a valid distribution
2. the small model trails the large model by enough for routing to buy anything
3. confidence separates errors from correct answers
4. the routed frontier beats random escalation
5. the prefix cache is actually firing

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes datasets
!pip -q install "pandas==2.2.3"
print("\nRESTART SESSION, then skip this cell")

In [ ]:
import os, sys, pathlib
os.environ.setdefault("CASCADE_ROOT", str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd()))

import time, gc, numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
from src.config import DATASETS
from src.datasets import labels_of, load_frozen
from src.scoring import (load_model, build_prefix, truncate, score_labels,
                         score_labels_naive, confidence_signals)
from src.routing import sweep, random_frontier, oracle_frontier, aucac, at_retention, ece

DATASET, N = "banking77", 200
assert torch.cuda.is_available(), "No GPU"
print("GPU:", torch.cuda.get_device_name(0))

## Run both tiers

Banking77 has 77 classes, which is genuinely hard for a 1.7B model, so there is
headroom for routing to work with.

In [ ]:
def run_tier(model_id, four_bit, n=N):
    tok, model = load_model(model_id, four_bit)
    df = load_frozen(DATASET, n); labels = labels_of(DATASET)
    ids = [tok(l, add_special_tokens=False, return_tensors="pt").input_ids for l in labels]
    rows, t0 = [], time.time()
    for i, r in df.iterrows():
        prefix = build_prefix(tok, truncate(tok, r.text, DATASETS[DATASET]["max_tokens"]), labels)
        p, norms, sums, n_pre, n_lab = score_labels(model, tok, prefix, ids, model.device)
        assert np.isfinite(p).all() and abs(p.sum()-1) < 1e-6, "invalid distribution"
        s = confidence_signals(p)
        rows.append({"instance_id": i, "gold": r.label, "pred": labels[s["pred_idx"]],
                     "c1_max_softmax": s["c1_max_softmax"], "c2_margin": s["c2_margin"],
                     "c3_norm_entropy": s["c3_norm_entropy"],
                     "prefix_tokens": n_pre, "label_tokens": n_lab})
        if (i+1) % 50 == 0:
            print(f"  {i+1}/{n}  {(time.time()-t0)/(i+1):.1f}s/item", flush=True)
    out = pd.DataFrame(rows)
    print(f"  accuracy {(out.pred==out.gold).mean():.3f}  ({(time.time()-t0)/60:.1f} min)")
    return out, tok, model, ids

print("small tier")
small, tok_s, model_s, ids_s = run_tier("Qwen/Qwen3-1.7B", False)

### Prefix-cache check

Expect 5x or better. Under 1.5x means caching is not firing.

In [ ]:
df = load_frozen(DATASET, 20); labels = labels_of(DATASET)
def timeit(fn):
    t = time.time()
    for x in df.text:
        fn(model_s, tok_s, build_prefix(tok_s, truncate(tok_s, x, 256), labels), ids_s, model_s.device)
    return time.time() - t
cached, naive = timeit(score_labels), timeit(score_labels_naive)
print(f"cached {cached:.1f}s vs naive {naive:.1f}s  ->  {naive/cached:.1f}x speedup")

del model_s, tok_s, ids_s; gc.collect(); torch.cuda.empty_cache()
print(f"freed, allocated {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
print("large tier")
large, *_ = run_tier("Qwen/Qwen3-14B", True)

## Verdict

In [ ]:
m = small.merge(large, on="instance_id", suffixes=("_s","_l"))
s_ok = (m.pred_s == m.gold_s).values.astype(int)
l_ok = (m.pred_l == m.gold_s).values.astype(int)
c_s = (1.7  * (m.prefix_tokens_s + m.label_tokens_s)).values.astype(float)
c_l = (14.0 * (m.prefix_tokens_l + m.label_tokens_l)).values.astype(float)

au = {c: roc_auc_score(s_ok, m[f"{c}_s"].values)
      for c in ["c1_max_softmax","c2_margin","c3_norm_entropy"]}
best = max(au, key=au.get)
f    = sweep(m[f"{best}_s"].values, s_ok, l_ok, c_s, c_l)
rand = random_frontier(s_ok, l_ok, c_s, c_l)
orac = oracle_frontier(s_ok, l_ok, c_s, c_l)
r95  = at_retention(f, l_ok.mean(), 0.95)
gap  = (l_ok.mean() - s_ok.mean()) * 100

checks = {
  "1 valid distributions"        : True,
  "2 gap between 8 and 30 pp"    : 8.0 <= gap <= 30.0,
  "3 confidence AUROC >= 0.60"   : max(au.values()) >= 0.60,
  "4 router beats random"        : aucac(f) > aucac(rand),
  "5 saving at 95% retention > 0": r95["saving"] > 0,
}
print(f"small {s_ok.mean():.3f} | large {l_ok.mean():.3f} | gap {gap:.1f} pp")
print("AUROC:", {k: round(v,3) for k,v in au.items()}, "-> best:", best)
print(f"ECE {ece(m.c1_max_softmax_s.values, s_ok):.3f}")
print(f"AUCAC router {aucac(f):.4f} random {aucac(rand):.4f} oracle {aucac(orac):.4f}")
print(f"at 95% retention: escalate {r95['escalation_rate']:.1%}, save {r95['saving']:.1%}")
print("-"*58)
for k,v in checks.items(): print(f"  {k:32s} {'PASS' if v else 'FAIL'}")
print(f"\nVERDICT: {'PASS' if all(checks.values()) else 'FAIL'}")

## Reading a failure

**Gap below 8 points.** The small model nearly matches the large one, so routing
has nothing to buy on this dataset. Check another before concluding anything.

**Gap above 30 points.** Check the prompt before blaming the model. A truncated
label list or a stray reasoning block both look exactly like this. Print ten raw
predictions and count how many distinct labels appear.

**AUROC below 0.60.** Confidence extraction is wrong, or predictions have
collapsed onto a handful of labels.

**Beats-random fails while AUROC is fine.** Sign convention. Escalation fires
below the threshold, so every signal must be oriented higher-is-more-confident.